# Dataset filtering

In [ ]:
import os
import pandas as pd

# spatial
import geopandas as gpd

from andeangc.config import find_repo_root, get_dir, get, get_version_dir
wd = find_repo_root()
os.chdir(wd)

path_final = get_version_dir('final')

# 1950_2024 — the period_q stamp carried by the artifact filenames
stamp_q = "_".join(get('period_q'))

## Data

In [ ]:
# read data from previous processing step
AndeanGC_data     = pd.read_csv(path_final / f'AndeanGC_data_{stamp_q}.csv').set_index('date')
AndeanGC_metadata = pd.read_csv(path_final / 'AndeanGC_metadata.csv').set_index('gauge_id')
AndeanGC_shape    = gpd.read_file(path_final / 'AndeanGC_shape.gpkg').set_index('gauge_id')
AndeanGC_shape["basin_area"] = AndeanGC_shape.to_crs(get('epsg_utm')).area / 1e6

## Basins selection

In [ ]:
# basins with glacier area > 0.1% (RGI v6.0)
rgi_version = get('rgi_version')
rgi = pd.concat([gpd.read_file(get_dir('glacier_outlines') / f"{rgi_version}_16.shp"), 
                 gpd.read_file(get_dir('glacier_outlines') / f"{rgi_version}_17.shp")])
rgi = rgi.rename(columns={"RGIId": "rgi_id"}).set_index("rgi_id")
rgi_union = rgi.union_all()

AndeanGC_shape = AndeanGC_shape[AndeanGC_shape.intersects(rgi_union)].copy()
AndeanGC_shape['glacier_area_RGI60'] = AndeanGC_shape.intersection(rgi_union).to_crs(epsg=get('epsg_utm')).area / 1e6
AndeanGC_shape = AndeanGC_shape.fillna(0)  # fill NaN values with 0

# > 0.1% glacier area
AndeanGC_shape['glacier_area_RGI60'] = (AndeanGC_shape.glacier_area_RGI60 * 100 / AndeanGC_shape.basin_area)
AndeanGC_shape = AndeanGC_shape[AndeanGC_shape.glacier_area_RGI60 > get('glacier_threshold')]

AndeanGC_data = AndeanGC_data[AndeanGC_shape.index]
AndeanGC_metadata = AndeanGC_metadata.loc[AndeanGC_shape.index]
AndeanGC_metadata = pd.concat([AndeanGC_metadata, AndeanGC_shape[["basin_area", "glacier_area_RGI60"]]], axis=1)

In [ ]:
# basins with more than 1 year of data (365 days)
AndeanGC_data = AndeanGC_data.interpolate(method='linear', limit=get('interpolation_limit')) # first interpolate (stations without data over weekends)
AndeanGC_metadata["days_w_data"] = AndeanGC_data.notna().sum()
AndeanGC_metadata = AndeanGC_metadata[AndeanGC_metadata.days_w_data > get('min_data_days')]

In [ ]:
# basins with "minimal" intervention (quality check is next step to remove more)
keywords_remove = "Embalse|Boca Toma|Central|Presa|Canal"
AndeanGC_metadata = AndeanGC_metadata[~AndeanGC_metadata["gauge_name"].str.contains(keywords_remove, case=False, na=False)]
AndeanGC_data = AndeanGC_data[AndeanGC_metadata.index]
AndeanGC_shape = AndeanGC_shape.loc[AndeanGC_metadata.index]

## Save

In [ ]:
AndeanGC_metadata.to_csv(path_final / 'AndeanGC_metadata.csv')
AndeanGC_data.to_csv(path_final / f'AndeanGC_data_{stamp_q}.csv')
AndeanGC_shape.to_file(path_final / 'AndeanGC_shape.gpkg')